# Activity 2 — Scraping Economic Data


| § | Technique | Source |
|---|---|---|
| 1 | API with a key | FRED (US macro) |
| 2 | HTML tables → `pd.read_html` | Jamaica — quarterly GDP |
| 3 | Browser headers + SSL bypass | Jamaica — stock exchange index |
| 4 | XML / SDMX → `ElementTree` | Jamaica — CPI, then Suriname — government operations |
| 5 | Direct file download | Suriname — central bank `.xlsx` |
| 6 | Link harvesting → BeautifulSoup | EIA Brent, US Census trade |

In [2]:
import io, os, re, json, glob, datetime, warnings
import requests
import pandas as pd
import numpy as np
from io import BytesIO, StringIO
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET

warnings.filterwarnings("ignore")           # keep the output readable
requests.packages.urllib3.disable_warnings()  # we deliberately skip some SSL checks below

# ---- Paths -----------------------------------------------------------------
# This notebook lives in workshop_code/activity/ (or solutions/), so the workshop
# folder is two levels up.
d          = os.getcwd()
PATH_RAW   = os.path.normpath(os.path.join(d, "..", "..", "raw"))
PATH_ACT2  = os.path.join(PATH_RAW, "act2")     # the bundled snapshots
os.makedirs(PATH_RAW, exist_ok=True)

# ---- The browser disguise --------------------------------------------------
# Many servers reject requests that do not look like a browser. Sending a
# User-Agent string is the single most useful line in this whole notebook.
hdr = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/58.0.3029.110 Safari/537")}

print("PATH_RAW  =", PATH_RAW)

PATH_RAW  = c:\Users\guerr\Dropbox\02_Work\Consulting\2026_NWCST_CAR_Research-Workshop\raw


# § 1 — FRED: API with a key

**Source shape:** the agency wants you to have the data and built a front door.

* [FRED](https://fred.stlouisfed.org/docs/api/fred/) — US macro and financial series
* [World Bank](https://datahelpdesk.worldbank.org/knowledgebase/topics/125589)
* [IMF](https://datahelp.imf.org/)

FRED needs a free key: register at
<https://fredredaccount.stlouisfed.org/apikeys> and paste it below. **Never commit a
key to a shared folder** — read it from an environment variable instead, as we do here.

In [5]:
# Get a free key at https://fred.stlouisfed.org  ->  My Account  ->  API Keys.
# Best practice: set it once as an environment variable rather than typing it into
# a notebook that you might later share.
FRED_API_KEY = "72304a91a13cc0ea6c610ae5845c7493"
series = ["SP500", "DJIA", "NASDAQCOM"]

df = pd.DataFrame()
for serie in series :
    #&realtime_end=9999-12-31
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={serie}&api_key={FRED_API_KEY}&file_type=json"
    response = requests.get(url)
    response = response.json()
    dfi = pd.DataFrame(response['observations'])
    dfi = dfi[['value', 'date']]
    dfi.set_index('date', inplace=True)
    dfi.index = pd.to_datetime(dfi.index)
    dfi.rename(columns = { "value" : f"fred_{serie}" }, inplace=True)
    dfi[f"fred_{serie}"] = dfi[f"fred_{serie}"].apply(pd.to_numeric, errors='coerce')
    df = df.merge(dfi, left_index=True, right_index=True, how='outer')
    df = pd.DataFrame(df.resample("D").mean())
    #df = pd.concat([df, dfi], axis=1)
    df.sort_index(inplace=True)
    
df.to_csv(f"{PATH_RAW}/fred_data.csv")
print(f"FRED Updated: {df.index.max()}")
df.tail(5)

FRED Updated: 2026-07-31 00:00:00


,fred_SP500,fred_DJIA,fred_NASDAQCOM
date,,,
2026-07-27,7413.18,52210.08,24932.08
2026-07-28,7428.78,52747.32,24876.91
2026-07-29,7316.15,51594.14,24442.94
2026-07-30,7437.63,52208.06,25122.18
2026-07-31,7489.72,52485.03,25373.85


# § 2 — HTML tables: `pd.read_html`

**Source shape:** the numbers are in a `<table>` on a web page.

`pd.read_html(url)` fetches a page and returns a **list** of every table it found.
One line of download, then all the work is cleaning.

Our example is **Jamaica's quarterly GDP** from the Statistical Institute of Jamaica.
The raw table arrives in the shape statistical agencies love and analysts hate:

* industries down the rows, `2015 Q1`, `2015 Q2`, ... across the columns
* a banner row of metadata sitting above the real header
* names full of `&`, `,` and double spaces
* numbers stored as text, with `-` meaning "missing"

We want the opposite: a `DatetimeIndex` down the rows, one column per variable.
Getting between the two is `melt` → clean → `pivot`, and it is the most reusable
20 lines in this notebook.

In [ ]:
JM_GDP_URL = "https://www.datazoa.com/data/table.asp?a=view&th=69E287AF0E&dzuuid=1835&uid=dzadmin"
# The official page is https://statinja.gov.jm/NationalAccounting/Quarterly/NewQuarterlyGDP.aspx
# but it is an ASP form that does not respond to a plain GET; datazoa republishes the
# same table in a scrapeable form.

df = pd.read_html(JM_GDP_URL)[0]
df = df.rename(columns={"Unnamed: 0": "variable"})

print("Jamaica GDP:")
print(f"   shape: {dfi.shape}")
df.iloc[:4, :5]

   live fetch OK
Jamaica GDP:
   shape: (14476, 1)


,variable,2024 Q4,2025 Q1,2025 Q2,2025 Q3
0,Total Value Added at Basic Prices,431334,436021,437133,438665
1,Agriculture Forestry & Fishing,33038,33882,34027,33460
2,Mining & Quarrying,5395,5882,5350,5158
3,Manufacturing,40104,40745,41792,42947


### Step 1 — tidy the variable names

Good practice everywhere: strip punctuation, lowercase, collapse whitespace, join
words with underscores. `Total Value Added at Basic Prices` becomes
`total_value_added_at_basic_prices` — something you can type without quoting.

Each `.str.<method>()` applies to every value in the column, and they chain.

In [14]:
# In Column variable: eliminate special symbols, extra spaces, use lowercase, and substitute spaces with underscores.
df["variable"] = df["variable"].str.replace(r"[^a-zA-Z0-9\s]", "", regex=True).str.lower().str.replace("  "," ").str.strip().str.replace(r"\s+", "_", regex=True)
df

,variable,2024 Q4,2025 Q1,2025 Q2,2025 Q3,2025 Q4,2026 Q1
0,total_value_added_at_basic_prices,431334,436021,437133,438665,406765,420272
1,agriculture_forestry_fishing,33038,33882,34027,33460,30617,30597
2,mining_quarrying,5395,5882,5350,5158,3658,4286
3,manufacturing,40104,40745,41792,42947,37010,41654
4,food_beverages_tobacco,25655,25659,26309,26802,23202,26308
5,other_manufacturing,14449,15086,15483,16145,13807,15346
6,electricity_water_supply_waste_management,11967,12063,11835,12207,10573,10771
7,construction,27197,27764,27988,28140,26931,27396
8,wholesale_retail_trade_repair_of_motor_vehicle...,72442,72383,72640,72178,71214,71906
9,accommodation_food_service_activities,25712,25362,25874,26115,17776,21119


### Step 2 — `melt`: wide to long

`pd.melt` "unpivots". Every date column becomes a row, so the table goes from
*18 rows × 41 columns* to *one row per (variable, date) pair*.

* `id_vars` — the column(s) to keep as identifiers
* `var_name` — name for the new column holding the old column **headers**
* `value_name` — name for the new column holding the **cell values**

In [15]:
# Reshape dataframe from wide to long format.
df = pd.melt(df, id_vars=["variable"], var_name="date", value_name="value")
# Transform value in numeric, make nan those values that cannot be converted.
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df

,variable,date,value
0,total_value_added_at_basic_prices,2024 Q4,431334.0
1,agriculture_forestry_fishing,2024 Q4,33038.0
2,mining_quarrying,2024 Q4,5395.0
3,manufacturing,2024 Q4,40104.0
4,food_beverages_tobacco,2024 Q4,25655.0
...,...,...,...
103,real_estate_business_activities,2026 Q1,52329.0
104,public_administration_defence,2026 Q1,27483.0
105,education_health_other_services,2026 Q1,44306.0
106,published_by_statistical_institute_of_jamaicap...,2026 Q1,NaN


### Step 3 — numbers, then dates

`errors="coerce"` is the important argument: instead of raising on `"-"` or `"n/a"`,
it writes `NaN` and moves on. Scraped tables are full of such sentinels.

Then the dates. They arrive as `2015 Q1`. Pandas cannot parse that, but the quarter
label maps cleanly onto the quarter's **first month** — `Q1 → 01`, `Q2 → 04`,
`Q3 → 07`, `Q4 → 10` — after which `"2015 01"` parses with `format="%Y %m"`.

In [16]:
# drop missing values
df.dropna(subset=["value"], inplace=True)
df

,variable,date,value
0,total_value_added_at_basic_prices,2024 Q4,431334.0
1,agriculture_forestry_fishing,2024 Q4,33038.0
2,mining_quarrying,2024 Q4,5395.0
3,manufacturing,2024 Q4,40104.0
4,food_beverages_tobacco,2024 Q4,25655.0
...,...,...,...
101,information_communication,2026 Q1,18042.0
102,financial_insurance_activities,2026 Q1,47173.0
103,real_estate_business_activities,2026 Q1,52329.0
104,public_administration_defence,2026 Q1,27483.0


In [17]:
# Transform date into datetime format. Date has format 'Q1 2020' but we want it to be the first day of the first month in the quarter
df["date"] = df["date"].str.replace("Q1", "01").str.replace("Q2", "04").str.replace("Q3", "07").str.replace("Q4", "10")
df["date"] = pd.to_datetime(df["date"], format="%Y %m")
df

,variable,date,value
0,total_value_added_at_basic_prices,2024-10-01,431334.0
1,agriculture_forestry_fishing,2024-10-01,33038.0
2,mining_quarrying,2024-10-01,5395.0
3,manufacturing,2024-10-01,40104.0
4,food_beverages_tobacco,2024-10-01,25655.0
...,...,...,...
101,information_communication,2026-01-01,18042.0
102,financial_insurance_activities,2026-01-01,47173.0
103,real_estate_business_activities,2026-01-01,52329.0
104,public_administration_defence,2026-01-01,27483.0


### Step 4 — `pivot`: long back to wide, the right way round

`pivot` is the inverse of `melt`. Now the *dates* become the index and the
*variables* become the columns — which is what every modelling tool expects.

In [18]:
# Pivot data to have variables as columns.
df = df.pivot(index="date", columns="variable", values="value")

df.rename(columns={'total_value_added_at_basic_prices':'gdp'}, inplace=True)
df.insert(0, "gdp", df.pop("gdp"))
df

variable,gdp,accommodation_food_service_activities,agriculture_forestry_fishing,construction,education_health_other_services,electricity_water_supply_waste_management,financial_insurance_activities,food_beverages_tobacco,information_communication,manufacturing,mining_quarrying,other_manufacturing,public_administration_defence,real_estate_business_activities,transport_storage,wholesale_retail_trade_repair_of_motor_vehicles_installation_of_machinery_equipment
date,,,,,,,,,,,,,,,,
2024-10-01,431334.0,25712.0,33038.0,27197.0,46026.0,11967.0,45302.0,25655.0,18472.0,40104.0,5395.0,14449.0,27353.0,53793.0,24533.0,72442.0
2025-01-01,436021.0,25362.0,33882.0,27764.0,46837.0,12063.0,46049.0,25659.0,18522.0,40745.0,5882.0,15086.0,27521.0,54132.0,24879.0,72383.0
2025-04-01,437133.0,25874.0,34027.0,27988.0,45855.0,11835.0,46881.0,26309.0,18129.0,41792.0,5350.0,15483.0,27612.0,54025.0,25125.0,72640.0
2025-07-01,438665.0,26115.0,33460.0,28140.0,45408.0,12207.0,47242.0,26802.0,18630.0,42947.0,5158.0,16145.0,27215.0,54155.0,25808.0,72178.0
2025-10-01,406765.0,17776.0,30617.0,26931.0,43822.0,10573.0,46364.0,23202.0,16162.0,37010.0,3658.0,13807.0,27891.0,52312.0,22436.0,71214.0
2026-01-01,420272.0,21119.0,30597.0,27396.0,44306.0,10771.0,47173.0,26308.0,18042.0,41654.0,4286.0,15346.0,27483.0,52329.0,23212.0,71906.0


# § 3 — When the server does not want to talk to you

**Source shape:** a normal HTML table, behind a server that blocks robots.

Two arguments to `requests.get` solve most of these:

* `headers=hdr` — the browser disguise from the setup cell.
* `verify=False` — skip SSL certificate validation. Plenty of government sites run
  expired or self-signed certificates, and Python refuses them by default where a
  browser would just warn you.

**`verify=False` is a real security trade-off**, not a formality: you lose the
guarantee that you are talking to who you think you are. Use it for public
statistical tables. Never use it for anything authenticated.

Example: the **Jamaica Stock Exchange** main index.

Suppose we try the same method as before

In [23]:
JSE_URL = "https://www.jamstockex.com/trading/indices/index-history/?indexCode=7&fromDate=2021-01-12"

df = pd.read_html(JSE_URL)[0]
df

HTTPError: HTTP Error 403: Forbidden

This error states that you are forbidden to get the data.
Basically, your ping was incorrect.
The solution is to mask your request:

In [ ]:
JSE_URL = "https://www.jamstockex.com/trading/indices/index-history/?indexCode=7&fromDate=2021-01-12"

r = requests.get(JSE_URL, timeout=60, verify=False, headers=hdr)
df = pd.read_html(r.text)[0]              # r.text = response decoded as string
df

,Date,Value,Change,Change (%),Volume Traded
0,Jul-31-2026,374012.30,3889.14,1.05%,41753131
1,Jul-30-2026,370123.16,2302.28,0.63%,30450116
2,Jul-29-2026,367820.88,-91.94,-0.02%,23316507
3,Jul-28-2026,367912.82,2136.74,0.58%,18813926
4,Jul-27-2026,365776.08,-184.16,-0.05%,13913064
...,...,...,...,...,...
1250,Aug-10-2021,418049.14,148.16,0.04%,5409606
1251,Aug-09-2021,417900.98,-3831.32,-0.91%,10019713
1252,Aug-05-2021,421732.30,74.68,0.02%,6485437
1253,Aug-04-2021,421657.62,-189.24,-0.04%,6341667


Basic cleaning

In [ ]:

df.columns = df.columns.astype(str).str.lower()
df = df[["date", "value", "volume traded"]].set_index("date")
df = df.rename(columns={"value": "jse_index", "volume traded": "jse_volume"})
# format="mixed" copes with more than one date layout in the same column
df.index = pd.to_datetime(df.index, format="mixed")
df.sort_index().apply(pd.to_numeric, errors="coerce")


print("Jamaica Stock Exchange:")
df.tail(3)

# § 4 — XML and SDMX

**Source shape:** a machine-readable statistical exchange format.

Many central banks publish under the IMF's **e-GDDS** standard, which means SDMX-XML.
It is *designed* for machines to easily access and clean the data. The structure is always:

```xml
<Series INDICATOR="PCPI_IX" ...>
    <Obs TIME_PERIOD="2020-01" OBS_VALUE="103.4"/>
    <Obs TIME_PERIOD="2020-02" OBS_VALUE="104.1"/>
</Series>
```

so you walk the `<Series>` elements, filter on the indicator you want, and read the
attributes off each `<Obs>`. `.//Series` is XPath for "every `<Series>` anywhere in
the document".

First: **Jamaica's CPI**.

In [24]:
JM_CPI_URL = "https://wups.statinja.gov.jm/wup/egddsfiles/CPI_Jamaica.xml"

r = requests.get(JM_CPI_URL, timeout=60, headers=hdr)
r.raise_for_status()                  # raise on any 4xx/5xx status code
root = ET.parse(BytesIO(r.content)).getroot()

rows = []
for s in root.findall(".//Series"):            # XPath: every <Series>, any depth
    if s.attrib.get("INDICATOR") == "PCPI_IX":  # consumer price index, all items
        for obs in s.findall("Obs"):            # direct <Obs> children
            v = obs.attrib.get("OBS_VALUE")
            if v is not None:
                rows.append((obs.attrib.get("TIME_PERIOD"), float(v)))

df = pd.DataFrame(rows, columns=["date", "cpi"])
df["date"] = pd.to_datetime(df["date"])
df.set_index("date").sort_index()

df

,date,cpi
0,2020-04-01,103.745131
1,2020-05-01,103.801412
2,2020-06-01,105.212719
3,2020-07-01,105.725541
4,2020-08-01,105.913309
...,...,...
70,2026-02-01,147.379332
71,2026-03-01,147.868385
72,2026-04-01,147.390568
73,2026-05-01,149.681003


# § 5 — Excel Files: a file at a fixed URL

**Source shape:** the agency just puts a spreadsheet on the server and leaves it there.

The **Central Bank van Suriname** publishes its whole real-sector database as one
`.xlsx` at a stable address. No parsing, no headers, no disguise — download the bytes,
wrap them in `BytesIO` so pandas can treat them as a file, and open.

`pd.ExcelFile` opens the workbook *without* reading every sheet, so you can look at
`.sheet_names` first and pick. That matters here: the file has dozens of sheets and
they get renamed every year (`22.1 GDP (real) 2015-2025`), so **match the sheet name
with a pattern rather than hardcoding it**. Hardcoded sheet names are the single most
common way these scripts break in January.

In [27]:
SR_XLSX_URL = "https://www.cbvs.sr/images/content/statistieken/Database/RealSectorStatistics.xlsx"

r = requests.get(SR_XLSX_URL, headers=hdr, timeout=120)
r.raise_for_status()
xls = pd.ExcelFile(BytesIO(r.content))         # bytes -> file-like -> workbook
xls

pandas reads the Excel but it is still not fully open: we need to locate the right sheet.

In [43]:
# Match the sheet, do not hardcode it: "22.1 GDP (real) 2015-2025" and friends.
cand = [s for s in xls.sheet_names if "gdp" in s.lower() and "real" in s.lower()]
print(f"   candidate sheets: {cand}")
df = pd.read_excel(xls, sheet_name=sorted(cand)[-1], skiprows=3)
df

   candidate sheets: ['22. GDP (real) 2006-2015', '22.1 GDP (real) 2015-2025']


,Unnamed: 0,Sector,2015,2016,2017,2018,2019,2020,2021,2022*,2023*,2024*,2025*(2),2026*(2)
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,"Agriculture, forestry and fishing",1822.952,1823.821,1706.858,1561.865,1268.001,1137.257,1051.756,1011.098,1004.499,914.415,886.447,904.071
3,NaN,Mining and quarrying (extraction),739.236,578.478,659.187,694.889,554.044,390.624,353.137,362.564,400.986,445.458,432.013,504.955
4,NaN,Manufacturing (inclusive milling and refining),2013.250,2104.166,2563.872,2713.866,2528.616,2566.891,2060.372,2098.265,2211.528,2077.542,1992.698,2076.494
5,NaN,"Electricity, gas, steam and air conditioning s...",318.288,235.553,292.278,289.681,307.744,246.226,251.982,257.510,260.923,263.966,271.885,280.042
6,NaN,"Water supply; sewerage, waste management and r...",18.554,18.741,18.595,18.571,18.225,18.216,20.322,21.220,21.737,21.634,22.093,22.756
7,NaN,Construction,1685.150,1229.798,1457.940,1454.390,1510.806,838.335,930.562,967.363,888.447,1061.334,1192.692,1271.990
8,NaN,Wholesale and retail trade; repair of motor ve...,2968.677,2965.107,2497.015,2752.394,2749.649,2602.279,2629.084,2696.129,2817.455,2986.502,3235.554,3394.995
9,NaN,Transportation and storage,695.719,534.023,647.500,717.115,694.693,497.571,498.414,509.323,550.069,654.243,678.877,713.140


Now that the file is open, basic data cleaning follows:

In [44]:
df = df.dropna(how="all", axis=0).dropna(how="all", axis=1)
df.columns = (df.columns.astype(str)
                .str.lower().str.strip()
                .str.replace(r"[^\w\s]", "", regex=True)
                .str.replace(r"\s+", "_", regex=True))
df = df.melt(id_vars="sector", var_name="year", value_name="value").dropna(subset=["value"])
df = df[df["sector"].astype(str).str.contains("Gross Domestic Product|GDP at market",
                                                case=False, na=False)]
df

,sector,year,value
19,GDP at market prices,2015,17514.647
45,GDP at market prices,2016,16654.387
71,GDP at market prices,2017,16915.201
97,GDP at market prices,2018,17752.211
123,GDP at market prices,2019,17959.484
149,GDP at market prices,2020,15090.422
175,GDP at market prices,2021,14722.912
201,GDP at market prices,2022,15077.178
227,GDP at market prices,2023,15441.532
253,GDP at market prices,2024,15707.274


In [45]:
df["date"] = pd.to_datetime(df["year"].astype(str).str[:4] + "-12-01", errors="coerce")
df = df[['date', 'value']]
df

,date,value
19,2015-12-01,17514.647
45,2016-12-01,16654.387
71,2017-12-01,16915.201
97,2018-12-01,17752.211
123,2019-12-01,17959.484
149,2020-12-01,15090.422
175,2021-12-01,14722.912
201,2022-12-01,15077.178
227,2023-12-01,15441.532
253,2024-12-01,15707.274


In [47]:
df = df.dropna(subset=["date"]).set_index("date")[["value"]].rename(columns={"value": "gdp"})
df

,gdp
date,
2015-12-01,17514.647
2016-12-01,16654.387
2017-12-01,16915.201
2018-12-01,17752.211
2019-12-01,17959.484
2020-12-01,15090.422
2021-12-01,14722.912
2022-12-01,15077.178
2023-12-01,15441.532


# § 6 — Link harvesting with BeautifulSoup

**Source shape:** the data file exists, but its URL changes every release.

You cannot hardcode `.../data_2026Q1.xlsx` because next quarter it is `2026Q2`.
Instead scrape the *page*, collect every `<a href="...">`, and filter for the one
you want. The page layout is far more stable than the filenames.

```python
soup  = BeautifulSoup(r.content, "html.parser")
links = [a.get("href") for a in soup("a") if a.get("href")]
link  = [l for l in links if "xls" in l][0]
```

Examples: **EIA Brent crude** (oil price)

In [48]:
url = "https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?n=PET&s=RBRTE&f=D"
r = requests.get(url, verify=False, headers=hdr, timeout=60).content
soup = BeautifulSoup(r, "html.parser")

links = [a.get("href") for a in soup("a") if a.get("href")]
link  = [l for l in links if "xls" in l][0].replace("../", "")   # fix relative path
print(f"   harvested link: {link}")

xls = pd.ExcelFile(f"https://www.eia.gov/dnav/pet/{link}")
sheet = [s for s in xls.sheet_names if "data" in s.lower()][0]
df = pd.read_excel(xls, sheet_name=sheet, skiprows=2)

df.columns = df.columns.str.lower()
df = df.set_index("date")
# rename with a lambda: applies to every column name
df = df.rename(columns=lambda x: "brent price" if "brent" in x else x)
df.index = pd.to_datetime(df.index)
df

   harvested link: hist_xls/RBRTEd.xls


,brent price
date,
1987-05-20,18.63
1987-05-21,18.45
1987-05-22,18.55
1987-05-25,18.60
1987-05-26,18.63
...,...
2026-07-21,93.85
2026-07-22,94.12
2026-07-23,105.32
